# Forecasting the September 2022 California Heat Wave — **AIFS v2 vs SFNO vs Aurora** (vs ERA5)

This notebook compares **three AI weather models** — ECMWF **AIFS v2**, NVIDIA/Makani **SFNO**, and
Microsoft **Aurora** — on the record-breaking **September 2022 California heat wave**
(event window ~Aug 31 – Sep 9 2022; peak ~Sep 6, when Sacramento hit 116 °F / 46.7 °C). All three run
via NVIDIA **earth2studio**, are initialized from the same **ERA5** state, and are verified against
ERA5 over the same valid times.

**Design**
- **Models:** a single ~12-day global deterministic forecast per model, all initialized
  **2022-08-30 00Z** (the day before onset, ~7-day lead to the peak). Each model is driven through the
  same `deterministic()` pipeline via a small `MODELS` registry; AIFS v2's already-cached run is reused.
- **Comparison variable:** **2 m temperature (`t2m`)** only — the field every model emits. Analyzed at
  **three scales**: global, continental US (CONUS), and California.
- **Verification:** ERA5 `t2m` from the Copernicus CDS, for the same valid times.
- **Heat-wave metric:** absolute **daily-maximum 2 m temperature** threshold (≥ 37.8 °C / 100 °F and
  ≥ 40 °C / 104 °F) sustained ≥ 3 consecutive days, plus per-model **bias / RMSE / MAE** vs ERA5.

**Per-model wrinkles (captured as registry flags; detail in §1 and §3).**
- **AIFS v2** needs 92 inputs, 16 of which are missing from earth2studio's CDS lexicon and are supplied
  by `aifs2_era5.patch_cds_lexicon_for_aifs2()` (`patch=True`). **SFNO** (73 vars) and **Aurora**
  (69 vars) resolve entirely from the stock lexicon, so they need no patch.
- **Aurora** runs on a 720×1440 grid (no south pole) vs the 721×1440 grid of AIFS v2 / SFNO / ERA5, so its
  output is regridded onto the ERA5 latitude grid before differencing (`regrid=True`).

**Two visualization modes.**
1. **Comparison grid** (§5) — peak-day maps with every model + ERA5 side-by-side per region, plus
   `model − ERA5` difference panels. Use it to *rank* the models spatially.
2. **Per-model deep dive** (§7) — for each model, a grouped figure of city daily-Tmax time series
   (model vs ERA5) and the model's error growth (bias/RMSE/MAE vs lead), for investigating one model in
   isolation. A cross-model RMSE/bias overlay (§6) summarizes "which model captured the heat dome best".

In [ ]:
# --- Environment (must be set BEFORE importing torch / earth2studio) ---
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUFILE_ENV_PATH_JSON"] = "/dev/null"

import gc, warnings
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
import xarray as xr
import torch
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from aifs2_era5 import patch_cds_lexicon_for_aifs2, ZERO_FILL_WAVE_VARS
from earth2studio.models.px import AIFS2, SFNO, Aurora
from earth2studio.lexicon import CDSLexicon
from earth2studio.data import CDS
from earth2studio.io import ZarrBackend
from earth2studio.run import deterministic
from earth2studio.utils.imports import OptionalDependencyError

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)

PROJECT  = Path("/home/jovyan/West-Coast")
DATA_DIR = PROJECT / "data"
FIG_DIR  = PROJECT / "figures"
for d in (DATA_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Event & forecast configuration ---
INIT_TIME     = np.datetime64("2022-08-30T00:00:00")  # day before onset; ~7-day lead to peak
FORECAST_DAYS = 12
NSTEPS        = FORECAST_DAYS * 4                       # 6-hourly steps -> 48
EVENT_START   = np.datetime64("2022-08-31")
EVENT_PEAK    = np.datetime64("2022-09-06")
EVENT_END     = np.datetime64("2022-09-09")
PRIMARY_VARS  = np.array(["t2m"])                       # the one field every model emits

# --- Model registry: the only things that differ between models ---
#   cls    : earth2studio prognostic class (same load + deterministic() pipeline for all)
#   label  : display name for plots
#   patch  : apply aifs2_era5.patch_cds_lexicon_for_aifs2() before init (AIFS v2 only)
#   regrid : interpolate the model's output onto the ERA5 grid before differencing
#            (Aurora is 720x1440; AIFS v2 / SFNO already match ERA5's 721x1440)
#   prefix : Zarr filename stem; "aifs" reuses the existing cached AIFS v2 forecast
#   extra  : earth2studio optional-dependency group needed to run it
# Each model needs its earth2studio optional deps installed, e.g. `uv add 'earth2studio[sfno,aurora]'`.
# Models whose deps are missing are skipped (with a warning) in the run cell below.
MODELS = {
    "AIFS2":  dict(cls=AIFS2,  label="AIFS v2", patch=True,  regrid=False, prefix="aifs",   extra="aifs2"),
    "SFNO":   dict(cls=SFNO,   label="SFNO",    patch=False, regrid=False, prefix="sfno",   extra="sfno"),
    "Aurora": dict(cls=Aurora, label="Aurora",  patch=False, regrid=True,  prefix="aurora", extra="aurora"),
}

# Regions: lat=(min,max); lon given in BOTH 0-360 (AIFS/ERA5 native) and -180..180 (plotting).
REGIONS = {
    "GLOBAL":     dict(lat=(-90, 90), lon360=(0, 360),   lon180=(-180, 180)),
    "CONUS":      dict(lat=(24, 50),  lon360=(235, 294), lon180=(-125, -66)),
    "CALIFORNIA": dict(lat=(32, 42),  lon360=(235, 246), lon180=(-125, -114)),
}

# Cities: name -> (lat, lon in -180..180)
CITIES = {
    "Sacramento":   (38.58, -121.49),
    "Fresno":       (36.74, -119.79),
    "Los Angeles":  (34.05, -118.24),
    "Death Valley": (36.46, -116.87),
    "Phoenix":      (33.45, -112.07),
    "Las Vegas":    (36.17, -115.14),
}

# Absolute daily-Tmax thresholds (deg C)
THRESHOLDS_C      = {"hot (100F)": 37.8, "extreme (104F)": 40.0}
HEATWAVE_MIN_DAYS = 3

print(f"init {INIT_TIME}  |  {NSTEPS} steps x 6h = {FORECAST_DAYS} days")
print("models:", ", ".join(f"{k} ({v['label']})" for k, v in MODELS.items()))

## 1 · Patch the ERA5/CDS lexicon and check model readiness

`patch_cds_lexicon_for_aifs2()` teaches earth2studio's CDS source the 16 AIFS-v2 inputs it is missing
(10 hPa level + snow depth from ERA5; the 11 wave inputs zero-filled). SFNO and Aurora need no patch —
their variables already resolve from the stock lexicon. After patching, the AIFS-v2 lexicon-gap probe
should report **no** missing variables. Model weights are loaded lazily inside `run_forecast()` (only
on a cache miss), so this cell stays cheap.

In [ ]:
# Patch the CDS lexicon for AIFS v2's extra inputs (idempotent; needed only for AIFS v2).
zeroed = patch_cds_lexicon_for_aifs2()
print("AIFS v2 lexicon patched; zero-filled wave inputs:", zeroed)

print("\nModels to compare:")
for k, s in MODELS.items():
    print(f"  {k:7s} {s['label']:9s}  patch={s['patch']!s:5s}  regrid={s['regrid']!s:5s}  prefix={s['prefix']}")

## 2 · Run the global forecasts (one per model)

`run_forecast(key, init_time)` drives any model in the registry through the identical earth2studio
`deterministic()` pipeline: it initializes from ERA5 (each model auto-fetches the input snapshots its
`input_coords()` requires — e.g. AIFS v2 and Aurora use a two-step `[-6h, 0h]` init) and integrates
`NSTEPS` 6-hour steps, saving `t2m` to a per-model Zarr store. To bound GPU memory, the model weights
are **loaded only on a cache miss and freed immediately after** — so at most one model is resident at a
time. AIFS v2's forecast is already cached and is reused as-is. The **first** SFNO / Aurora run downloads
the checkpoint and the full ERA5 initial state from CDS (slow, then cached); Aurora is the heavy one.
Any model whose optional deps are missing or broken is skipped (with a warning) rather than failing the
whole run, so the rest of the notebook adapts to whatever models are actually installed.

In [ ]:
def run_forecast(key, init_time, nsteps=NSTEPS, variables=PRIMARY_VARS, overwrite=False):
    '''Deterministic global forecast for model `key` from an ERA5 init -> path to a Zarr store.

    The model weights are loaded only on a cache miss and freed afterwards, so at most one model
    is resident on the GPU at a time (Aurora is large).'''
    spec = MODELS[key]
    tag      = np.datetime_as_string(np.datetime64(init_time), unit="D").replace("-", "")
    out_path = DATA_DIR / f"{spec['prefix']}_forecast_init{tag}.zarr"
    if out_path.exists() and not overwrite:
        print(f"{key:7s} using cached forecast: {out_path.name}")
        return out_path
    if spec["patch"]:
        patch_cds_lexicon_for_aifs2()                 # AIFS v2 only; idempotent
    print(f"{key:7s} loading model + running {nsteps} steps ...")
    model = spec["cls"].load_model(spec["cls"].load_default_package()).to(DEVICE)
    deterministic(
        time=[np.datetime64(init_time)],
        nsteps=nsteps,
        prognostic=model,
        data=CDS(cache=True),
        io=ZarrBackend(str(out_path), backend_kwargs={"overwrite": True}),
        output_coords=OrderedDict({"variable": np.asarray(variables)}),
        device=DEVICE,
    )
    del model                                          # free before the next model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"{key:7s} forecast -> {out_path.name}")
    return out_path

# Missing/broken optional deps (incl. transitive import errors) -> skip the model instead of crashing.
LOAD_DEP_ERRORS = (OptionalDependencyError, ImportError, AttributeError)

# Run each model; skip (and drop from MODELS) any whose deps are missing/broken, so the rest of the
# notebook adapts to whatever is installed (e.g. AIFS2 + Aurora when SFNO/makani is unavailable).
paths, _skipped = {}, []
for k in list(MODELS):
    try:
        paths[k] = run_forecast(k, INIT_TIME)
    except LOAD_DEP_ERRORS as e:
        _skipped.append(k)
        print(f"{k:7s} SKIPPED ({type(e).__name__}) — needs earth2studio[{MODELS[k]['extra']}] with a "
              f"working dependency stack. First line: {str(e).splitlines()[0][:120]}")
for k in _skipped:
    del MODELS[k]

print("\nactive models:", list(MODELS))
print("forecast Zarr stores:")
for k, p in paths.items():
    print(f"  {k:7s} {p}")

## 3 · Load forecasts, fetch ERA5, regrid Aurora, align

For each model we read its `t2m`, attach a **valid-time** axis (`init + lead`), and convert to °C. We
fetch ERA5 `t2m` once at those valid times (it shares the AIFS v2 / SFNO 0.25° 721×1440 grid). **Aurora**
lives on a 720×1440 grid (no south pole), so its field is interpolated onto the ERA5 latitude grid
(`regrid=True`) before comparison — see `load_forecast_t2m()` below for the interpolation detail; the
boundary effect at the pole is immaterial for a California heat-wave. Finally every model and ERA5 are
aligned on the common valid-time axis. All series are sampled 6-hourly, so the daily-Tmax comparison is
apples-to-apples across models.

In [4]:
def load_forecast_t2m(path, init_time, regrid_to=None):
    '''Open a forecast Zarr -> t2m DataArray (valid_time, lat, lon) in deg C.

    If `regrid_to` (a DataArray carrying lat/lon coords) is given, interpolate onto its grid.
    Used for Aurora's 720x1440 grid -> the ERA5/AIFS 721x1440 grid (1-D lat interp; lon is identical).'''
    ds = xr.open_zarr(str(path))
    da = ds["t2m"]
    for d in ("batch", "time"):                       # squeeze singleton init dims
        if d in da.dims:
            da = da.isel({d: 0}, drop=True)
    valid = np.datetime64(init_time) + da["lead_time"].values
    da = da.assign_coords(lead_time=valid).rename(lead_time="valid_time")
    da = (da - 273.15).assign_attrs(units="degC")
    if regrid_to is not None:
        da = da.interp(lon=regrid_to["lon"])                                   # identical grid in practice
        da = da.interp(lat=regrid_to["lat"], kwargs={"fill_value": "extrapolate"})  # 720 -> 721, fill pole
    return da

def fetch_era5_t2m(times):
    '''ERA5 t2m at the given valid times via earth2studio's CDS data source
    -> DataArray (valid_time, lat, lon) in deg C. Same backend as the model init
    (idiomatic, self-caching) and grid-matched (721x1440) to the forecasts.'''
    src = CDS(cache=True)
    da = src(time=np.asarray(times), variable="t2m")     # (time, variable, lat, lon), Kelvin
    da = da.isel(variable=0, drop=True).rename(time="valid_time")
    return (da - 273.15).assign_attrs(units="degC")

# Valid times come from any model (all share init + steps); use a non-regrid model as the reference.
REF_KEY     = next(k for k in MODELS if not MODELS[k]["regrid"])
valid_times = load_forecast_t2m(paths[REF_KEY], INIT_TIME)["valid_time"].values
era5_t2m    = fetch_era5_t2m(valid_times)

# Load every model's t2m (regridding onto the ERA5 grid where flagged).
fc = {k: load_forecast_t2m(paths[k], INIT_TIME,
                           regrid_to=(era5_t2m if MODELS[k]["regrid"] else None))
      for k in MODELS}

# Align all models + ERA5 on the common (valid_time, lat, lon) axes.
aligned = xr.align(*fc.values(), era5_t2m, join="inner")
fc = {k: aligned[i] for i, k in enumerate(fc)}
era5_t2m = aligned[-1]

print("aligned grid:", dict(era5_t2m.sizes), "|", str(era5_t2m["valid_time"].values[0])[:13],
      "->", str(era5_t2m["valid_time"].values[-1])[:13])
for k in fc:
    print(f"  {k:7s} {MODELS[k]['label']:9s} {dict(fc[k].sizes)}  NaNs={bool(np.isnan(fc[k].values).any())}")
print("  ERA5    truth      ", dict(era5_t2m.sizes), " NaNs=", bool(np.isnan(era5_t2m.values).any()))

2026-06-16 22:22:46.595 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-30 00:00:00 with 1 levels


2026-06-16 22:22:47.087 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-30 06:00:00 with 1 levels
2026-06-16 22:22:47.119 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-30 12:00:00 with 1 levels
2026-06-16 22:22:47.150 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-30 18:00:00 with 1 levels
2026-06-16 22:22:47.181 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-31 00:00:00 with 1 levels
2026-06-16 22:22:47.211 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-31 06:00:00 with 1 levels
2026-06-16 22:22:47.242 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-08-31 12:00:00 

2026-06-16 22:22:47.303 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-01 00:00:00 with 1 levels
2026-06-16 22:22:47.334 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-01 06:00:00 with 1 levels
2026-06-16 22:22:47.364 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-01 12:00:00 with 1 levels
2026-06-16 22:22:47.395 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-01 18:00:00 with 1 levels
2026-06-16 22:22:47.426 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-02 00:00:00 with 1 levels
2026-06-16 22:22:47.458 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-02 06:00:00 

2026-06-16 22:22:47.519 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-02 18:00:00 with 1 levels
2026-06-16 22:22:47.549 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-03 00:00:00 with 1 levels
2026-06-16 22:22:47.580 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-03 06:00:00 with 1 levels
2026-06-16 22:22:47.611 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-03 12:00:00 with 1 levels
2026-06-16 22:22:47.641 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-03 18:00:00 with 1 levels
2026-06-16 22:22:47.672 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-04 00:00:00 

2026-06-16 22:22:47.733 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-04 12:00:00 with 1 levels
2026-06-16 22:22:47.764 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-04 18:00:00 with 1 levels
2026-06-16 22:22:47.794 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-05 00:00:00 with 1 levels
2026-06-16 22:22:47.825 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-05 06:00:00 with 1 levels
2026-06-16 22:22:47.855 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-05 12:00:00 with 1 levels
2026-06-16 22:22:47.886 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-05 18:00:00 

2026-06-16 22:22:47.947 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-06 06:00:00 with 1 levels
2026-06-16 22:22:47.978 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-06 12:00:00 with 1 levels
2026-06-16 22:22:48.009 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-06 18:00:00 with 1 levels
2026-06-16 22:22:48.039 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-07 00:00:00 with 1 levels
2026-06-16 22:22:48.070 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-07 06:00:00 with 1 levels
2026-06-16 22:22:48.101 | INFO     | earth2studio.data.cds:_fetch_process:177 - Fetching CDS grib file for variable: 2m_temperature at 2022-09-07 12:00:00 

... (2 outputs left out)

aligned grid: {'valid_time': 49, 'lat': 721, 'lon': 1440} | 2022-08-30T00 -> 2022-09-11T00


  AIFS2   AIFS v2   {'valid_time': 49, 'lat': 721, 'lon': 1440}  NaNs=False


  SFNO    SFNO      {'valid_time': 49, 'lat': 721, 'lon': 1440}  NaNs=False


  Aurora  Aurora    {'valid_time': 49, 'lat': 721, 'lon': 1440}  NaNs=False
  ERA5    truth       {'valid_time': 49, 'lat': 721, 'lon': 1440}  NaNs= False


## 4 · Helpers: regional cropping, longitude conversion, heat-wave metrics

In [ ]:
def crop_bbox(da, lat_range, lon360_range, lat_name="lat", lon_name="lon"):
    '''Slice a DataArray (lon in 0-360, lat descending) to a bbox given in 0-360 lon.'''
    lo_lat, hi_lat = min(lat_range), max(lat_range)
    da = da.sel({lat_name: slice(hi_lat, lo_lat)})         # lat is descending
    lo_lon, hi_lon = lon360_range
    if lo_lon <= hi_lon:
        return da.sel({lon_name: slice(lo_lon, hi_lon)})
    lon = da[lon_name]
    return da.sel({lon_name: lon[(lon >= lo_lon) | (lon <= hi_lon)]})   # dateline wrap

def to_180(da, lon_name="lon"):
    '''Convert lon 0-360 -> -180..180 and sort (for cartopy plotting).'''
    da = da.assign_coords({lon_name: (((da[lon_name] + 180) % 360) - 180)})
    return da.sortby(lon_name)

def daily_tmax(da, time_name="valid_time"):
    '''6-hourly t2m -> daily maximum; renames the day axis to 'day'.'''
    out = da.resample({time_name: "1D"}).max()
    return out.rename({time_name: "day"})

def area_weights(da, lat_name="lat"):
    w = np.cos(np.deg2rad(da[lat_name]))
    return w.where(np.isfinite(w), 0.0)

def exceedance_mask(tmax_daily, thresh_c):
    return tmax_daily >= thresh_c

def heatwave_daycount(mask, day_name="day", min_days=HEATWAVE_MIN_DAYS):
    '''Per gridpoint: number of days belonging to a run of >= min_days exceedance days.'''
    a = mask.transpose(day_name, ...)
    arr = a.values.astype(np.int32)
    D = arr.shape[0]
    # Run length at each day = (consecutive exceedance days ending here) + (starting here) - 1,
    # computed via one forward and one backward cumulative pass instead of a per-gridpoint loop.
    f = np.zeros_like(arr); b = np.zeros_like(arr)
    f[0] = arr[0]
    for d in range(1, D):  f[d] = arr[d] * (f[d - 1] + 1)
    b[-1] = arr[-1]
    for d in range(D - 2, -1, -1):  b[d] = arr[d] * (b[d + 1] + 1)
    runlen = np.where(arr > 0, f + b - 1, 0)
    daycount = (runlen >= min_days).sum(axis=0)
    dims = a.dims[1:]
    return xr.DataArray(daycount, dims=dims, coords={d: a.coords[d] for d in dims})

def exceedance_area_frac(mask, lat_name="lat"):
    '''Per-day cos(lat)-weighted fraction of the region exceeding the threshold.'''
    w = area_weights(mask, lat_name)
    return (mask * w).sum(("lat", "lon")) / (w * xr.ones_like(mask.isel(day=0))).sum(("lat", "lon"))

def error_metrics_per_day(model_daily, era5_daily, lat_name="lat"):
    '''cos(lat)-weighted bias / RMSE / MAE between a model daily-Tmax field and ERA5, per day.'''
    w = area_weights(model_daily, lat_name)
    diff = model_daily - era5_daily
    norm = (w * xr.ones_like(diff)).sum(("lat", "lon"))
    bias = (diff * w).sum(("lat", "lon")) / norm
    rmse = np.sqrt(((diff**2) * w).sum(("lat", "lon")) / norm)
    mae  = (np.abs(diff) * w).sum(("lat", "lon")) / norm
    return xr.Dataset({"bias": bias, "rmse": rmse, "mae": mae})

# Daily-Tmax fields used throughout: one per model + ERA5 truth.
tmax     = {k: daily_tmax(fc[k]) for k in MODELS}
era5_tmax = daily_tmax(era5_t2m)
days     = era5_tmax["day"].values
xdays    = pd.to_datetime(days)           
print("daily Tmax days:", [str(d)[:10] for d in days])
print("models:", list(tmax))

## 5 · Comparison grid — peak-day maps, all models vs ERA5

Peak-day 2 m daily-Tmax at each scale. **Top row:** every model + ERA5 truth on a shared color scale.
**Bottom row:** each `model − ERA5` difference on a shared symmetric scale (the ERA5 column has no diff).
These are ~7-day-lead forecasts, so meaningful differences are expected — this view ranks the models
spatially at a glance.

In [ ]:
def comparison_grid(region):
    '''Peak-day Tmax maps: top row = each model + ERA5; bottom row = model - ERA5 differences.'''
    info = REGIONS[region]
    peak = days[np.argmin(np.abs(days - EVENT_PEAK))]

    # crop+convert once per model; ERA5 too
    e = to_180(crop_bbox(era5_tmax.sel(day=peak), info["lat"], info["lon360"]))
    mflds = {MODELS[k]["label"]: to_180(crop_bbox(tmax[k].sel(day=peak), info["lat"], info["lon360"]))
             for k in MODELS}
    top   = {**mflds, "ERA5": e}
    diffs = {lab: fld - e for lab, fld in mflds.items()}

    vmin = float(min(f.min() for f in top.values()))
    vmax = float(max(f.max() for f in top.values()))
    dmax = max(float(np.nanmax(np.abs(d.values))) for d in diffs.values()) or 1.0

    ncol = len(top)                                   # models + ERA5
    fig, axes = plt.subplots(2, ncol, figsize=(5.1 * ncol, 9.0),
                             subplot_kw={"projection": ccrs.PlateCarree()})

    def style(ax):
        ax.set_extent([*info["lon180"], *info["lat"]], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, lw=0.6)
        ax.add_feature(cfeature.BORDERS, lw=0.4)
        if region != "GLOBAL":
            ax.add_feature(cfeature.STATES, lw=0.4)

    for ax, (name, fld) in zip(axes[0], top.items()):
        m = ax.pcolormesh(fld["lon"], fld["lat"], fld, cmap="inferno", vmin=vmin, vmax=vmax,
                          shading="auto", transform=ccrs.PlateCarree())
        style(ax)
        ax.set_title(name, fontsize=11)
        fig.colorbar(m, ax=ax, shrink=0.7, label="degC")

    # ERA5 column has no diff, left blank
    for ax, (name, d) in zip(axes[1], diffs.items()):
        m = ax.pcolormesh(d["lon"], d["lat"], d, cmap="RdBu_r", vmin=-dmax, vmax=dmax,
                          shading="auto", transform=ccrs.PlateCarree())
        style(ax)
        ax.set_title(f"{name} − ERA5", fontsize=11)
        fig.colorbar(m, ax=ax, shrink=0.7, label="degC")
    for ax in axes[1][len(diffs):]:
        ax.axis("off")

    fig.suptitle(f"{region} · peak-day Tmax · {str(peak)[:10]}", y=1.0, fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"maps_peakday_{region.lower()}.png", dpi=110, bbox_inches="tight")
    plt.show()

for r in ("GLOBAL", "CONUS", "CALIFORNIA"):
    comparison_grid(r)

## 6 · Cross-model skill — RMSE / bias vs lead (California)

The headline comparison: cos(lat)-weighted daily-Tmax error over California for each model, day by day
(the dotted line marks the Sep 6 peak). Lower RMSE and near-zero bias ⇒ the model that best captured the
heat dome at this ~7-day lead. The printed table gives the peak-day numbers per model.

In [ ]:
reg  = REGIONS["CALIFORNIA"]
e_ca = crop_bbox(era5_tmax, reg["lat"], reg["lon360"])
peak = days[np.argmin(np.abs(days - EVENT_PEAK))]

fig, (ax_r, ax_b) = plt.subplots(1, 2, figsize=(14, 4.8))
for k in MODELS:
    err = error_metrics_per_day(crop_bbox(tmax[k], reg["lat"], reg["lon360"]), e_ca)
    ax_r.plot(xdays, err["rmse"].values, marker="o", ms=3, label=MODELS[k]["label"])
    ax_b.plot(xdays, err["bias"].values, marker="o", ms=3, label=MODELS[k]["label"])
for ax, ttl in ((ax_r, "RMSE"), (ax_b, "Bias")):
    ax.axvline(xpeak, color="k", lw=0.8, ls=":")
    ax.set_title(ttl); ax.set_ylabel("degC")
    ax.grid(alpha=0.3); ax.legend(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax_b.axhline(0, color="k", lw=0.6)
fig.suptitle("California Tmax skill vs ERA5", fontsize=13)
fig.tight_layout()
fig.savefig(FIG_DIR / "skill_comparison_california.png", dpi=110, bbox_inches="tight")
plt.show()

# Headline: California peak-day skill per model.
print(f"California peak-day ({str(peak)[:10]}) daily-Tmax skill vs ERA5:")
for k in MODELS:
    s = error_metrics_per_day(crop_bbox(tmax[k], reg["lat"], reg["lon360"]), e_ca).sel(day=peak)
    print(f"  {MODELS[k]['label']:9s} bias={float(s['bias']):+.2f} degC   "
          f"RMSE={float(s['rmse']):.2f} degC   MAE={float(s['mae']):.2f} degC")

## 7 · Per-model deep dives — model vs ERA5

One grouped figure per model so you can investigate a single model against ERA5 in isolation:
**left** = daily-Tmax time series at each city (model solid, ERA5 dashed, one color per city);
**right** = that model's California error growth (bias / RMSE / MAE vs lead). The dotted line marks Sep 6.

In [ ]:
def city_series(daily, lat, lon):
    '''Nearest-gridpoint daily-Tmax series at a city (lon given in -180..180).'''
    return daily.sel(lat=lat, lon=lon % 360, method="nearest")

def model_vs_era5_panel(key):
    '''Grouped per-model figure: city time series (model vs ERA5) + California error growth.'''
    label  = MODELS[key]["label"]
    md, ed = tmax[key], era5_tmax
    reg    = REGIONS["CALIFORNIA"]
    err    = error_metrics_per_day(crop_bbox(md, reg["lat"], reg["lon360"]),
                                   crop_bbox(ed, reg["lat"], reg["lon360"]))

    fig, (axL, axR) = plt.subplots(1, 2, figsize=(15, 5))
    cmap = plt.get_cmap("tab10")
    for i, (city, (clat, clon)) in enumerate(CITIES.items()):
        c = cmap(i % 10)
        axL.plot(xdays, city_series(md, clat, clon).values, color=c, lw=1.9, label=city)
        axL.plot(xdays, city_series(ed, clat, clon).values, color=c, lw=1.2, ls="--")
    axL.axvline(xpeak, color="k", lw=0.8, ls=":")
    axL.set_title("City Tmax  (solid: model, dashed: ERA5)")
    axL.set_ylabel("daily Tmax (degC)"); axL.grid(alpha=0.3); axL.legend(fontsize=8, ncol=2)
    axL.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

    for name in ("bias", "rmse", "mae"):
        axR.plot(xdays, err[name].values, marker="o", ms=3, label=name.upper())
    axR.axhline(0, color="k", lw=0.6); axR.axvline(xpeak, color="k", lw=0.8, ls=":")
    axR.set_title("Error vs ERA5  (California)")
    axR.set_ylabel("degC"); axR.grid(alpha=0.3); axR.legend()
    axR.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

    fig.suptitle(f"{label} vs ERA5", fontsize=13)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"deepdive_{key.lower()}.png", dpi=110, bbox_inches="tight")
    plt.show()

for k in MODELS:
    model_vs_era5_panel(k)